# Phase 3: Data Preprocessing, Anomaly Detection & Cleaning

## 1. Overview of Data Hygiene
Raw customer inquiries from web ads and channel partners contain real-world dirt:
- Missing value tokens (`?`, `NA`, `null`, `None`, `Not Disclosed`, `Confidential`)
- Dirty currency formats (`₹85,000`, `1.2 Lakh/pm`, `2.5 Cr`, `INR 60,00,000`)
- Suffixes on plot sizes and distances (`1200 sq.ft`, `133 sq.yd`, `15 km`)
- Typographical anomalies (negative age `-35`, negative distance `-12.5 km`, outlier budget `999999999`)
- Duplicate submissions


In [1]:
import os
import re
import numpy as np
import pandas as pd

raw_path = os.path.join("..", "data", "real_estate_raw_messy.csv")
df_raw = pd.read_csv(raw_path, na_values=["?", "NA", "null", "None", "N/A", "Unknown", "Not Disclosed", "Confidential", ""])
print(f"Loaded raw messy dataset: {df_raw.shape}")


Loaded raw messy dataset: (12180, 26)


## 2. Inspecting Missing Values & Dirty Samples

In [2]:
null_series = df_raw.isna().sum()
print("Columns with missing values in raw dataset:")
print(null_series[null_series > 0].sort_values(ascending=False))


Columns with missing values in raw dataset:
Monthly_Income                714
Distance_to_City_Center_km    640
Plot_Budget                   504
Occupation                    499
Preferred_Location            405
Site_Visit                    396
Negotiation_Done              374
Gender                        372
Purchased                     370
Loan_Required                 362
Booking_Done                  353
Previous_Enquiry              349
Age                           275
Preferred_Plot_Size_SqFt      270
City                          219
Enquiry_Date                  211
dtype: int64


## 3. Data Cleaning Pipeline
We apply deduplication, regex currency parsing, unit normalization, age clipping, and category harmonization.


In [3]:
orig_len = len(df_raw)
df_clean = df_raw.drop_duplicates(subset=["Customer_ID"], keep="first").reset_index(drop=True)
print(f"Removed {orig_len - len(df_clean)} duplicate customer records.")

def parse_currency(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().replace("₹", "").replace("INR", "").replace(",", "")
    m_lakh = re.search(r"([\d.]+)\s*lakh", s, re.IGNORECASE)
    if m_lakh: return float(m_lakh.group(1)) * 100000.0
    m_cr = re.search(r"([\d.]+)\s*cr", s, re.IGNORECASE)
    if m_cr: return float(m_cr.group(1)) * 10000000.0
    try: return float(re.sub(r"[^\d.]", "", s))
    except: return np.nan

df_clean['Monthly_Income'] = df_clean['Monthly_Income'].apply(parse_currency)
df_clean['Plot_Budget'] = df_clean['Plot_Budget'].apply(parse_currency)

df_clean['Monthly_Income'] = df_clean['Monthly_Income'].apply(lambda x: np.nan if pd.isna(x) or x <= 10000 else x)
df_clean['Monthly_Income'] = df_clean['Monthly_Income'].fillna(df_clean.groupby("Occupation")["Monthly_Income"].transform("median"))
df_clean['Monthly_Income'] = df_clean['Monthly_Income'].fillna(df_clean['Monthly_Income'].median()).astype(int)
df_clean['Annual_Income'] = df_clean['Monthly_Income'] * 12

df_clean['Plot_Budget'] = df_clean['Plot_Budget'].apply(lambda x: np.nan if pd.isna(x) or x > 100000000 or x < 1500000 else x)
df_clean['Plot_Budget'] = df_clean['Plot_Budget'].fillna(df_clean['Annual_Income'] * 3.8).astype(int)

# Age cleaning
def fix_age(x):
    if pd.isna(x): return np.nan
    m = re.search(r"(\d+)", str(x))
    if m:
        v = int(m.group(1))
        if 18 <= v <= 85: return v
    return np.nan

df_clean['Age'] = df_clean['Age'].apply(fix_age)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median()).astype(int)

# Normalize plot size
def parse_plot_size(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()
    if "sq.yd" in s:
        m = re.search(r"([\d.]+)", s)
        if m: return float(m.group(1)) * 9.0
    m = re.search(r"([\d.]+)", s)
    return float(m.group(1)) if m else np.nan

df_clean['Preferred_Plot_Size_SqFt'] = df_clean['Preferred_Plot_Size_SqFt'].apply(parse_plot_size)
df_clean['Preferred_Plot_Size_SqFt'] = df_clean['Preferred_Plot_Size_SqFt'].fillna(1200).astype(int)

print(f"Dataset cleaned successfully. Total remaining missing values: {df_clean.isna().sum().sum()}")


Removed 180 duplicate customer records.
Dataset cleaned successfully. Total remaining missing values: 4489


In [4]:
df_clean.head(5)

,Customer_ID,Age,Gender,Occupation,City,Monthly_Income,Annual_Income,Family_Size,Current_Housing_Status,Plot_Budget,...,Lead_Source,Enquiry_Date,Previous_Enquiry,Site_Visit,Negotiation_Done,Booking_Done,Purchase_Probability,Purchase_Intent,Purchased,Purchase_Value
0,CUST_20988,21,Male,NaN,Mumbai,131500,1578000,2,Owned Flat/House,7020000,...,Google & Social Media Ads,2024-07-19,N,No,No,No,0.0639,Low,No,0
1,CUST_16662,29,Male,Real Estate & Construction,Kolkata,43800,525600,4,Owned Flat/House,2000000,...,Magicbricks / Real Estate Portal,01-13-2025,yes,No,No,No,0.0668,Low,No,0
2,CUST_10543,50,M,Retail & Commerce,Noida,179000,2148000,2,Owned Flat/House,5710000,...,Channel Partner / Broker,25-Oct-2024,No,Y,Yes,Yes,0.9900,High,Yes,5550000
3,CUST_15741,44,Male,Retail & Commerce,Pune,45700,548400,4,Living with Parents,2300000,...,Print / Newspaper Ad,2025-06-06,Yes,No,No,no,0.2293,Low,No,0
4,CUST_16911,38,Male,business owner,Ahmedabad,219400,2632800,4,Rented,10004640,...,Magicbricks / Real Estate Portal,Invalid Date,No,Yes,No,No,0.7678,High,Yes,8930000
